In [5]:
t = 3
cutcoef_U1 = [[0] * (t+1) for _ in range(2)]
print(cutcoef_U1)

[[0, 0, 0, 0], [0, 0, 0, 0]]


In [1]:
import gurobipy as gp
from gurobipy import GRB

# Step 1: Create an empty model
model = gp.Model()

# Step 2: Read the LP file
model = gp.read("Error_Backward_t7_k(188.09, 1).lp")  # Correct way
# Step 3: Optimize the model
model.optimize()

# Step 4: Check the result
if model.status == GRB.OPTIMAL:
    print("Optimal objective value:", model.objVal)
    
    # Optional: print variable values
    for v in model.getVars():
        print(f"{v.varName}: {v.x}")
else:
    print("Optimization was not successful. Status code:", model.status)


Set parameter Username
Set parameter LicenseID to value 2767213
Academic license - for non-commercial use only - expires 2027-01-18
Read LP format model from file Error_Backward_t7_k(188.09, 1).lp
Reading time = 0.04 seconds
: 1145 rows, 172 columns, 54181 nonzeros
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i9-12900HK, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 1145 rows, 172 columns and 54181 nonzeros
Model fingerprint: 0x6d937c65
Coefficient statistics:
  Matrix range     [2e-09, 8e+03]
  Objective range  [3e-01, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 9e+08]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 913 rows and 122 columns
Presolve time: 0.01s
Presolved: 232 rows, 50 columns, 6308 nonzeros

Iteration    Obj

In [2]:
import numpy as np

objs = [14199747.33759272, 24614005.75514008, 15644322.52600235, 13840096.2747652, 15644322.52600235]
nbOS = len(objs)

mean_obj = np.mean(objs)
std_obj = np.std(objs)   # sample standard deviation
ci_low = mean_obj - 1.96 * std_obj / np.sqrt(nbOS)
ci_high = mean_obj + 1.96 * std_obj / np.sqrt(nbOS)
CI = mean_obj - ci_low

print(f"\nTwo-Stage OOS Evaluation Results")
print(f"μ ± 1.96*σ/√NS = {mean_obj:.2f} ± {CI:.2f}")
print(f"95% Confidence Interval = [{ci_low:.2f}, {ci_high:.2f}]")




Two-Stage OOS Evaluation Results
μ ± 1.96*σ/√NS = 16788498.88 ± 3489714.90
95% Confidence Interval = [13298783.98, 20278213.78]


In [5]:
import numpy as np
import pandas as pd
# Show all rows (be careful with very large DataFrames!)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Fake network processor object with needed attributes
class DummyNetworkProcessor:
    def __init__(self):
        self.V = 21  # total number of nodes (indexed 0 to 20)
        self.V3 = [14, 15, 16, 17, 18, 19, 20]  # PoD node indices
        self.T_Aft = [10, 11, 12, 13, 14, 15]   # post-landfall time periods

network_processor = DummyNetworkProcessor()

# Test state
k_t = (65.0, 3)   # Example state: cat = 4
std_pod = 1       # STD for time distribution

def msp_post_landfall_demand(k_t, network_processor):
        """
        Generate deterministic PoD-level demand for t > t_L (i.e., post-landfall),
        based on the hurricane category from the Markov state k_t at landfall time.
        """
        # Load pod info from Excel
        pod_info_df = pd.read_excel("Data/PoD location information.xlsx", sheet_name="Sheet1")
        
        # Extract hurricane category from Markov state
        cat = int(k_t[1])

        # Initialize local demand dictionary
        demand = {}

        if cat == 0 or pd.isna(cat):
            # No demand if hurricane category is 0 or NaN
            for i in network_processor.V3:
                for t in network_processor.T_Aft:
                    demand[(i, t)] = 0.0
        else:
            # Retrieve demand-related columns
            demand_col = f"I*p_c (cat {cat})"
            evac_col = "% 1- (Likely to evacuate)"
            total_demand = pod_info_df[demand_col] * pod_info_df[evac_col]
            total_demand = total_demand.to_numpy()

            # Gaussian-weighted temporal distribution
            mean = np.mean(network_processor.T_Aft)
            std = std_pod
            weights = np.array([
                np.exp(-((t - mean) ** 2) / (2 * std ** 2)) for t in network_processor.T_Aft
            ])
            weights /= weights.sum()

            # Assign demand to PoD nodes
            for idx, i in enumerate(network_processor.V3):
                for w_idx, t in enumerate(network_processor.T_Aft):
                    if (i, t) not in demand:
                        demand[(i, t)] = total_demand[idx] * weights[w_idx]

        # Assign zero demand to all non-PoD nodes
        for i in range(network_processor.V):
            if i not in network_processor.V3:
                for t in network_processor.T_Aft:
                    if (i, t) not in demand:
                        demand[(i, t)] = 0.0

        return demand

# Call the function
demand_dict = msp_post_landfall_demand(k_t, network_processor)
print(demand_dict)

df = pd.DataFrame([(i, t, d) for (i, t), d in demand_dict.items()],
                  columns=["Node", "Time", "Demand"])
df.sort_values(["Node", "Time"]).reset_index(drop=True)


{(14, 10): np.float64(99.27927366175558), (14, 11): np.float64(733.5801225478002), (14, 12): np.float64(1994.0775168404446), (14, 13): np.float64(1994.0775168404446), (14, 14): np.float64(733.5801225478002), (14, 15): np.float64(99.27927366175558), (15, 10): np.float64(753.4209378141279), (15, 11): np.float64(5567.069575617533), (15, 12): np.float64(15132.864065168345), (15, 13): np.float64(15132.864065168345), (15, 14): np.float64(5567.069575617533), (15, 15): np.float64(753.4209378141279), (16, 10): np.float64(293.9466174880321), (16, 11): np.float64(2171.9880467099783), (16, 12): np.float64(5904.07563900199), (16, 13): np.float64(5904.07563900199), (16, 14): np.float64(2171.9880467099783), (16, 15): np.float64(293.9466174880321), (17, 10): np.float64(216.11857787817445), (17, 11): np.float64(1596.9122959629437), (17, 12): np.float64(4340.857675758882), (17, 13): np.float64(4340.857675758882), (17, 14): np.float64(1596.9122959629437), (17, 15): np.float64(216.11857787817445), (18, 10

,Node,Time,Demand
0,0,10,0.000000
1,0,11,0.000000
2,0,12,0.000000
3,0,13,0.000000
4,0,14,0.000000
5,0,15,0.000000
6,1,10,0.000000
7,1,11,0.000000
8,1,12,0.000000
9,1,13,0.000000


In [ ]:
import json
import numpy as np
import pandas as pd
import time
np.random.seed()

def load_transition_dict(path):
    with open(path, "r") as f:
        return json.load(f)

def build_node_lists(transition_dict):
    nodeLists = {}
    for t in transition_dict:
        nodeLists[int(t)] = [eval(s) for s in transition_dict[t].keys()]
    return nodeLists

def MC_sample(t, current_state, transition_dict):
    """ Draw one next state given current time t and current_state """

    # Convert keys to strings
    t_key     = str(t)
    state_key = str(current_state)
    # Get all possible transitions
    probs_dict = transition_dict[t_key][state_key]
    # Extract valid next states and probabilities
    states_str = [s for s, p in probs_dict.items() if p > 0]
    weights    = [p for s, p in probs_dict.items() if p > 0]
    states = [eval(s) for s in states_str] # Convert state strings to tuples
    idx    = np.random.choice(len(states), p=weights) # Randomly sample next state
    return states[idx]

def create_OSpaths(nbOS, T_Bef, k_init, transition_dict, output_file):
    """
    Generate Out-of-Sample (OOS) paths using transition probabilities.
    Saves a CSV file where each cell contains a stringified tuple representing a state.
    """
    T = len(T_Bef)
    OS_paths = np.full((nbOS, T), None, dtype=object)

    # Initialize first state for all paths
    OS_paths[:, 0] = k_init

    # Generate paths using MC sampling
    for s in range(nbOS):
        for t in range(1, T):
            prev_state = OS_paths[s, t-1]
            next_state = MC_sample(t-1, prev_state, transition_dict)
            OS_paths[s, t] = next_state

    # Save the paths as CSV
    df = pd.DataFrame(OS_paths)
    df.to_csv(output_file, index=False)

    print(f"[INFO] OOS paths generated and saved to: {output_file}")
    return OS_paths

In [ ]:
import numpy as np
xval = [[0] * 5 for _ in range(3)]
xval1 = np.zeros((3, 5))
print(xval)
print(xval1)

[[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [ ]:
V = [0, 1, 2]
A = [(0, 1), (0, 2), (1, 2)]

for (i, j) in A:
    print("homa")

for i in range(len(V)):
    for j in range(len(V)):
        if (i, j) in A:
            print("milad")



homa
homa
homa
milad
milad
milad


In [11]:
f1_prev = {}
for t in range(3):
    for i in range(2):
        for j in range(2):
            f1_prev[i, j, t] = 0.0

print(f1_prev)

for t in range(3):
    for i in range(2):
        for j in range(2):
            key = (i, j, t)
            f1_prev[(key)] = 7.0
print(f1_prev)

{(0, 0, 0): 0.0, (0, 1, 0): 0.0, (1, 0, 0): 0.0, (1, 1, 0): 0.0, (0, 0, 1): 0.0, (0, 1, 1): 0.0, (1, 0, 1): 0.0, (1, 1, 1): 0.0, (0, 0, 2): 0.0, (0, 1, 2): 0.0, (1, 0, 2): 0.0, (1, 1, 2): 0.0}
{(0, 0, 0): 7.0, (0, 1, 0): 7.0, (1, 0, 0): 7.0, (1, 1, 0): 7.0, (0, 0, 1): 7.0, (0, 1, 1): 7.0, (1, 0, 1): 7.0, (1, 1, 1): 7.0, (0, 0, 2): 7.0, (0, 1, 2): 7.0, (1, 0, 2): 7.0, (1, 1, 2): 7.0}


In [9]:
T = 10
for t in range(9, 0, -1):
    print(t)

9
8
7
6
5
4
3
2
1


In [ ]:
import json

with open("Data/pi_mssp_d_ST_track_15.json", "r") as f:
    transition_dict = json.load(f)

nodeLists = {int(t): [eval(s) for s in trans.keys()] 
             for t, trans in transition_dict.items()}

# All Markov states reachable at stage t
print(nodeLists[2])

[(-32.52, 0), (-32.52, 1), (-32.52, 2), (-32.52, 3), (-32.52, 4), (-32.52, 5), (10.24, 0), (10.24, 1), (10.24, 2), (10.24, 3), (10.24, 4), (10.24, 5), (34.28, 0), (34.28, 1), (34.28, 2), (34.28, 3), (34.28, 4), (34.28, 5)]


In [12]:
import json

with open("Data/pi_mssp_d_ST_track_15.json", "r") as f:
    data = json.load(f)

print("Top-level keys (stages):", list(data.keys())[:10])
first_stage = list(data.keys())[0]
print("Sample stage:", first_stage)
print("Sample states at stage 0:", list(data[first_stage].keys())[:5])

first_state = list(data[first_stage].keys())[0]
print("Sample transitions from", first_state, ":", data[first_stage][first_state])


Top-level keys (stages): ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Sample stage: 0
Sample states at stage 0: ['(0.0, 0)']
Sample transitions from (0.0, 0) : {'(-23.55, 0)': 0.256, '(-23.55, 1)': 0.203, '(-23.55, 2)': 0.023, '(-23.55, 3)': 0.0, '(-23.55, 4)': 0.0, '(-23.55, 5)': 0.0, '(23.55, 0)': 0.275, '(23.55, 1)': 0.219, '(23.55, 2)': 0.024, '(23.55, 3)': 0.0, '(23.55, 4)': 0.0, '(23.55, 5)': 0.0}


In [14]:
import json

def load_transition_dict(path):
    with open(path, "r") as f:
        return json.load(f)

def build_node_lists(transition_dict):
    nodeLists = {}
    for t in transition_dict:
        nodeLists[int(t)] = [eval(s) for s in transition_dict[t].keys()]
    return nodeLists

# Load transition dictionary and build nodeLists
transition_dict = load_transition_dict("Data/pi_mssp_d_ST_track_15.json")
nodeLists = build_node_lists(transition_dict)

state_to_idx = {}
idx = 0
for t in nodeLists:
    for k in nodeLists[t]:
        state_to_idx[k] = idx
        idx += 1

K = len(state_to_idx)
print(K)

t = 8
n = list(transition_dict[str(t)].keys())[0]
print("Current state at t =", n)
print("Reachable next states at t+1:")
print(transition_dict[str(t)][n])



343
Current state at t = (-526.81, 0)
Reachable next states at t+1:
{'(-1110.62, 0)': 0.249, '(-1110.62, 1)': 0.198, '(-1110.62, 2)': 0.022, '(-1110.62, 3)': 0.0, '(-1110.62, 4)': 0.0, '(-1110.62, 5)': 0.0, '(-360.84, 0)': 0.266, '(-360.84, 1)': 0.211, '(-360.84, 2)': 0.024, '(-360.84, 3)': 0.0, '(-360.84, 4)': 0.0, '(-360.84, 5)': 0.0, '(-160.18, 0)': 0.016, '(-160.18, 1)': 0.013, '(-160.18, 2)': 0.001, '(-160.18, 3)': 0.0, '(-160.18, 4)': 0.0, '(-160.18, 5)': 0.0, '(-69.33, 0)': 0.0, '(-69.33, 1)': 0.0, '(-69.33, 2)': 0.0, '(-69.33, 3)': 0.0, '(-69.33, 4)': 0.0, '(-69.33, 5)': 0.0, '(-26.59, 0)': 0.0, '(-26.59, 1)': 0.0, '(-26.59, 2)': 0.0, '(-26.59, 3)': 0.0, '(-26.59, 4)': 0.0, '(-26.59, 5)': 0.0, '(-8.75, 0)': 0.0, '(-8.75, 1)': 0.0, '(-8.75, 2)': 0.0, '(-8.75, 3)': 0.0, '(-8.75, 4)': 0.0, '(-8.75, 5)': 0.0, '(12.43, 0)': 0.0, '(12.43, 1)': 0.0, '(12.43, 2)': 0.0, '(12.43, 3)': 0.0, '(12.43, 4)': 0.0, '(12.43, 5)': 0.0, '(35.9, 0)': 0.0, '(35.9, 1)': 0.0, '(35.9, 2)': 0.0, '(35.9,

In [22]:
import numpy as np

def MC_sample_json(t, current_state, transition_dict):
    """
    Return ONE next state given (t , current_state).

    Parameters
    ----------
    t : int
        Current time step (0-based, as in your forward pass).
    current_state : tuple
        Current Markov state, e.g. (23.55, 1).
    transition_dict : dict
        Loaded JSON dictionary of transition probabilities.

    Returns
    -------
    tuple
        Next state sampled according to transition probabilities.
    """
    # Keys in the JSON are strings
    t_key       = str(t)
    state_key   = str(current_state)

    probs_dict  = transition_dict[t_key][state_key]          # { "(next_state)": prob, ... }
    states_str  = [s for s, p in probs_dict.items() if p > 0]
    weights     = [p for s, p in probs_dict.items() if p > 0]

    states      = [eval(s) for s in states_str]              # back to tuple form
    idx         = np.random.choice(len(states), p=weights)   # Monte-Carlo draw
    return states[idx]


In [33]:
# --- 2. Set known values -------------------------------------------------
t              = 1              # current time step
current_state  = (-23.55, 0)       # deterministic starting state at t = 0

# --- 3. Call the single-step sampler ------------------------------------
next_state = MC_sample_json(t, current_state, transition_dict)

print(f"Current state (t={t}): {current_state}")
print(f"Sampled next state   : {next_state}")

Current state (t=1): (-23.55, 0)
Sampled next state   : (10.24, 0)
